In [ ]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## preprocess and embedding

In [ ]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [ ]:
control_key = "is_control"
condition_keys = "target_gene"
condition_rep_keys = "gene_embeddings"
random_seed = 42
dataset_name = "ArcVirtualCell"

In [ ]:
filePath = './data/raw/adata_Training.h5ad'
adata = sc.read_h5ad(filePath)
#adata = adata[adata.obs.sample(frac=0.1, random_state=42).index].to_memory()
print(adata)

In [ ]:
adata.obs[control_key] = (adata.obs[condition_keys] == "non-targeting")
gene_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

In [ ]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.1
gene_list = list(gene_list)
zero_shot = False

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    adata_pert = adata[adata.obs[control_key] == False].copy()
    y = adata_pert.obs[condition_keys].astype(str).values
    idx = np.arange(adata_pert.n_obs)
    train_idx, test_idx = train_test_split(
        idx,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata_pert[train_idx].copy()
    adata_test = adata_pert[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    print(gene_list)
    del adata, adata_pert
else:
    # 按基因分割 zero-shot
    n_test = max(1, int(len(gene_list) * test_ratio))
    test_gene = rng.choice(gene_list, size=n_test, replace=False).tolist()
    print(test_gene)
    train_gene = [g for g in gene_list if g not in test_gene]
    print(train_gene)
    adata_control = adata[adata.obs[control_key]==True].copy() # control的target_gene是non-targeting
    adata_train = adata[adata.obs[condition_keys].isin(train_gene)].copy() 
    adata_test = adata[adata.obs[condition_keys].isin(test_gene)].copy()
    del adata

In [ ]:
sample_rep = "X_pca" 
#sample_rep = "X_scVI" 
#sample_rep = "X_flatvi"
#sample_rep = "X_scVI_linear"
n_comps = 256
n_hidden = 2048
condition_rep_dict = pd.read_pickle("./data/processed/vcc_data_target_genes_embedding.pkl")
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/scvi_random{random_seed}_ncomps{n_comps}_hidden{n_hidden}"
flatvi_save_path = f"./data/processed/model/flatvi_random{random_seed}_ncomps{n_comps}_hidden{n_hidden}"
load_embedding_model = True
if sample_rep == "X_scVI":
    adata_control.layers["counts"] = adata_control.X.copy()
    adata_train.layers["counts"] = adata_train.X.copy()
    if adata_test is not None:
        adata_test.layers["counts"] = adata_test.X.copy()

    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{scvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{scvi_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{scvi_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep in ["X_flatvi","X_scVI_linear"]:
    adata_control.layers["counts"] = adata_control.X.copy()
    adata_train.layers["counts"] = adata_train.X.copy()
    if adata_test is not None:
        adata_test.layers["counts"] = adata_test.X.copy()


In [ ]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    scvi_save_path = scvi_save_path,
    flatvi_save_path = flatvi_save_path,
    condition_rep_dict = condition_rep_dict,
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
if sample_rep == "X_pca":
    sample_rep_scaled = sample_rep + "_scaled" # 额 别忘了
else:
    sample_rep_scaled = sample_rep

In [ ]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep_scaled}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [ ]:
print(adata_control)
print(adata_train)
print(adata_test)